# Inpaint Backgrounds

## Setup

In [8]:
import sys
from pathlib import Path
from typing import Any
from PIL import Image

# Add the project root to sys.path to allow imports from avm package
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from avm.inpainting import (
    MODELS,
    detect_device,
    get_prompt,
    build_model,
    inpaint_background,
    get_model_config_defaults
)

## Constants

In [9]:
IMAGES_DIR = '../data/bg1k_imgs/'  # Contains images named <ID>.png e.g., 0.png, 1.png, ...
MASKS_DIR = '../data/bg1k_masks/'  # Contains masks named <ID>_mask.png e.g., 0_mask.png
INFO_TXT_PATH = '../data/bg1k_info.txt'  # Each line: <image name>\t<category>
MODEL_ID = MODELS['stable-diffusion-xl-1.0-inpainting-0.1']
OUT_IMAGES_DIR = f'../data/bg1k_out_imgs/{MODEL_ID}/'  # Stores output images named <ID>.png


SEED = 42
PROMPT_OVERRIDE = ''  # If non-empty, overrides category prompt for all images

# Config overrides
CONFIG_OVERRIDES: dict[str, int | float] = {
    # Example overrides:
    # "num_inference_steps": 30,
}

# Filtering / execution controls
IMAGE_IDS = None  # Example: {0, 1, 2}
MAX_IMAGES: int | None = None  # None means no limit
OVERWRITE = False
DEVICE = None  # None -> auto detect via detect_device()

images_dir = Path(IMAGES_DIR)
masks_dir = Path(MASKS_DIR)
info_txt_path = Path(INFO_TXT_PATH)
out_images_dir = Path(OUT_IMAGES_DIR)
out_images_dir.mkdir(parents=True, exist_ok=True)

In [10]:
def _load_info_records(path: Path):
    if not path.exists():
        raise FileNotFoundError(f'Info file not found: {path}')

    records: list[dict[str, Any]] = []
    with path.open('r', encoding='utf-8') as handle:
        for line_no, raw_line in enumerate(handle, start=1):
            line = raw_line.strip()
            if not line:
                continue

            parts = line.split('\t')
            if len(parts) < 2:
                print(f'Skipping malformed line {line_no}: {line!r}')
                continue

            image_name = parts[0].strip()
            category = parts[1].strip()
            image_id = Path(image_name).stem

            image_path = images_dir / image_name
            mask_path = masks_dir / f'{image_id}_mask.png'
            out_path = out_images_dir / f'{image_id}.png'

            records.append({
                'image_id': image_id,
                'image_name': image_name,
                'category': category,
                'image_path': image_path,
                'mask_path': mask_path,
                'out_path': out_path,
            })
    return records

records = _load_info_records(info_txt_path)
print(f'Loaded {len(records)} record(s) from {info_txt_path}.')

if records:
    print('First record:', records[0])

Loaded 1000 record(s) from ../data/bg1k_info.txt.
First record: {'image_id': '0', 'image_name': '0.png', 'category': 'refrigerator', 'image_path': PosixPath('../data/bg1k_imgs/0.png'), 'mask_path': PosixPath('../data/bg1k_masks/0_mask.png'), 'out_path': PosixPath('../data/bg1k_out_imgs/diffusers/stable-diffusion-xl-1.0-inpainting-0.1/0.png')}


In [11]:
# Apply filtering based on MAX_IMAGES and IMAGE_IDS
if MAX_IMAGES is not None:
    records = records[:MAX_IMAGES]

if IMAGE_IDS is not None:
    records = [r for r in records if int(r['image_id']) in IMAGE_IDS]

## Generate

In [12]:
model = build_model(MODEL_ID, DEVICE or detect_device())
config = {
    **get_model_config_defaults(MODEL_ID),
    **CONFIG_OVERRIDES,
}
print(f'Loaded model {MODEL_ID} with config: {config}')

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

The config attributes {'decay': 0.9999, 'inv_gamma': 1.0, 'min_decay': 0.0, 'optimization_step': 37000, 'power': 0.6666666666666666, 'update_after_step': 0, 'use_ema_warmup': False} were passed to UNet2DConditionModel, but are not expected and will be ignored. Please verify your config.json configuration file.


Loaded model diffusers/stable-diffusion-xl-1.0-inpainting-0.1 with config: {'strength': 1.0, 'guidance_scale': 7.5, 'num_inference_steps': 50}


In [13]:
processed = 0
skipped_existing = 0
missing_inputs = 0
failed = 0

for idx, record in enumerate(records, start=1):
    image_path = record['image_path']
    mask_path = record['mask_path']
    out_path = record['out_path']
    category = record['category']

    if not image_path.exists() or not mask_path.exists():
        missing_inputs += 1
        print(f'[{idx}/{len(records)}] Missing input(s): image={image_path.exists()} mask={mask_path.exists()} -> {record["image_id"]}')
        continue

    if out_path.exists() and not OVERWRITE:
        skipped_existing += 1
        print(f'[{idx}/{len(records)}] Skipping existing output: {out_path.name}')
        continue

    category_prompt = get_prompt(category.strip())
    rendered_prompt = PROMPT_OVERRIDE.strip() or category_prompt

    print(f'[{idx}/{len(records)}] Processing {record["image_id"]}.png...')
    input_image = Image.open(image_path)
    mask_image = Image.open(mask_path)

    output_image = inpaint_background(
        model,
        input_image,
        mask_image,
        rendered_prompt,
        seed=SEED,
        config=config
    )

    output_image.save(out_path)
    processed += 1
    print(f'[{idx}/{len(records)}] Saved: {out_path}')

print('---')
print(f'Done. processed={processed}, skipped_existing={skipped_existing}, missing_inputs={missing_inputs}, failed={failed}')

[1/1000] Skipping existing output: 0.png
[2/1000] Skipping existing output: 1.png
[3/1000] Skipping existing output: 2.png
[4/1000] Skipping existing output: 3.png
[5/1000] Skipping existing output: 4.png
[6/1000] Skipping existing output: 5.png
[7/1000] Skipping existing output: 6.png
[8/1000] Skipping existing output: 7.png
[9/1000] Skipping existing output: 8.png
[10/1000] Skipping existing output: 9.png
[11/1000] Skipping existing output: 10.png
[12/1000] Skipping existing output: 11.png
[13/1000] Skipping existing output: 12.png
[14/1000] Skipping existing output: 13.png
[15/1000] Skipping existing output: 14.png
[16/1000] Skipping existing output: 15.png
[17/1000] Skipping existing output: 16.png
[18/1000] Skipping existing output: 17.png
[19/1000] Skipping existing output: 18.png
[20/1000] Skipping existing output: 19.png
[21/1000] Processing 20.png...


  0%|          | 0/50 [00:00<?, ?it/s]

[21/1000] Saved: ../data/bg1k_out_imgs/diffusers/stable-diffusion-xl-1.0-inpainting-0.1/20.png
[22/1000] Processing 21.png...


  0%|          | 0/50 [00:00<?, ?it/s]

[22/1000] Saved: ../data/bg1k_out_imgs/diffusers/stable-diffusion-xl-1.0-inpainting-0.1/21.png
[23/1000] Processing 22.png...


...

[98/1000] Saved: ../data/bg1k_out_imgs/diffusers/stable-diffusion-xl-1.0-inpainting-0.1/97.png
[99/1000] Processing 98.png...


  0%|          | 0/50 [00:00<?, ?it/s]

[99/1000] Saved: ../data/bg1k_out_imgs/diffusers/stable-diffusion-xl-1.0-inpainting-0.1/98.png
[100/1000] Processing 99.png...


  0%|          | 0/50 [00:00<?, ?it/s]

KeyboardInterrupt: 